# Biohub Local Method Compare

This notebook compares method outputs on train samples with GEFF labels. It does not reveal the Kaggle hidden score; it gives a consistent local sparse-label score for our baseline and any stronger notebook output saved as `submission.csv`.

Use it in Colab after Drive is mounted. Put downloaded artifact zips under `external_archives/` if you want the notebook to unpack/check them.

In [ ]:
from pathlib import Path
import json, math, os, shutil, subprocess, sys, time, zipfile

import numpy as np
import pandas as pd

try:
    from google.colab import drive
    drive.mount('/content/drive')
except Exception:
    pass

PROJECT_DIR = Path('/content/drive/MyDrive/Biohub - Cell Tracking During Development')
if not PROJECT_DIR.exists():
    PROJECT_DIR = Path.cwd()
BASE_DIR = PROJECT_DIR / 'data'
REPORT_DIR = PROJECT_DIR / 'reports'
ARCHIVE_DIR = PROJECT_DIR / 'external_archives'
ARTIFACT_DIR = PROJECT_DIR / 'artifacts'
REPORT_DIR.mkdir(parents=True, exist_ok=True)
ARCHIVE_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

# Keep this small first. Increase after the flow is verified.
N_VALIDATION_SAMPLES = 6
MATCH_MAX_DISTANCE_UM = 7.0
VOXEL_SCALE_UM = np.array([1.625, 0.40625, 0.40625], dtype=float)

# Optional: generate our classical baseline local submission for the same validation samples.
RUN_BASELINE_PREDICTION = False
BASELINE_MAX_TIMEPOINTS = 100

# Put method CSV outputs here. Baseline can be generated by this notebook; DeepCenter should be copied from its local validation run.
METHOD_SUBMISSIONS = {
    'baseline': REPORT_DIR / 'baseline_local_validation_submission.csv',
    'deepcenter': REPORT_DIR / 'deepcenter_local_validation_submission.csv',
}

print('PROJECT_DIR:', PROJECT_DIR)
print('BASE_DIR:', BASE_DIR)
print('REPORT_DIR:', REPORT_DIR)
print('ARCHIVE_DIR:', ARCHIVE_DIR)
print('ARTIFACT_DIR:', ARTIFACT_DIR)

## Artifact Zip Check

For Colab, upload these zips to Drive under `external_archives/`:

- `archive.zip`: DeepCenter center prior
- `archive (1).zip`: tracking support pack with repo, wheels, and weights
- `archive (2).zip`: optional figure pack, not needed for scoring

In [ ]:
def zip_contains(zip_path: Path, needle: str) -> bool:
    try:
        with zipfile.ZipFile(zip_path) as zf:
            return any(name.endswith(needle) or needle in name for name in zf.namelist())
    except Exception:
        return False

def find_artifact_zips():
    search_dirs = [ARCHIVE_DIR, PROJECT_DIR, Path('/content')]
    zips = []
    for root in search_dirs:
        if root.exists():
            zips.extend(sorted(root.glob('*.zip')))
    support = None
    center = None
    figures = None
    for zp in zips:
        if support is None and zip_contains(zp, 'repo/scripts/predict_unet_transformer.py'):
            support = zp
        if center is None and zip_contains(zp, 'weights/full_frame_center/checkpoint_last.pt'):
            center = zp
        if figures is None and zip_contains(zp, 'biohub_lineage.png'):
            figures = zp
    return support, center, figures, zips

def extract_if_needed(zip_path: Path | None, target_dir: Path, marker: str):
    if zip_path is None:
        print('missing zip for', target_dir.name)
        return False
    if (target_dir / marker).exists():
        print('already extracted:', target_dir)
        return True
    target_dir.mkdir(parents=True, exist_ok=True)
    print('extracting', zip_path.name, '->', target_dir)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(target_dir)
    print('done')
    return (target_dir / marker).exists()

support_zip, center_zip, figures_zip, all_zips = find_artifact_zips()
print('found zips:', [p.name for p in all_zips])
print('support_zip:', support_zip)
print('center_zip:', center_zip)
print('figures_zip optional:', figures_zip)

SUPPORT_DIR = ARTIFACT_DIR / 'biohub-tracking-support-pack-50ep-v1'
CENTER_DIR = ARTIFACT_DIR / 'biohub-deepcenter-unet3d-center-prior-v1'
extract_if_needed(support_zip, SUPPORT_DIR, 'repo/scripts/predict_unet_transformer.py')
extract_if_needed(center_zip, CENTER_DIR, 'weights/full_frame_center/checkpoint_last.pt')
print('support ok:', (SUPPORT_DIR / 'ARTIFACT_MANIFEST.json').exists())
print('center ok:', (CENTER_DIR / 'ARTIFACT_MANIFEST.json').exists())

## Dependencies

If Colab misses packages, install from the support-pack wheels first. This avoids relying on internet when possible.

In [ ]:
INSTALL_MISSING_DEPS = False

def missing_modules(names):
    import importlib.util
    return [name for name in names if importlib.util.find_spec(name) is None]

needed = ['zarr', 'scipy', 'skimage']
missing = missing_modules(needed)
print('missing:', missing)
if missing and INSTALL_MISSING_DEPS:
    wheel_dir = SUPPORT_DIR / 'wheels'
    if wheel_dir.exists():
        cmd = [sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links', str(wheel_dir), 'zarr', 'scipy', 'scikit-image', 'numcodecs', 'blosc2']
        print('running:', ' '.join(cmd))
        subprocess.check_call(cmd)
    else:
        cmd = [sys.executable, '-m', 'pip', 'install', 'zarr', 'scipy', 'scikit-image', 'numcodecs', 'blosc2']
        print('running:', ' '.join(cmd))
        subprocess.check_call(cmd)

missing_after = missing_modules(needed)
if missing_after:
    print('Still missing deps. Set INSTALL_MISSING_DEPS=True and rerun this cell:', missing_after)
else:
    print('Dependency check OK')

In [ ]:
def sample_id_from_path(path: Path) -> str:
    name = path.name
    return name[:-5] if name.endswith('.zarr') else path.stem

def load_validation_samples():
    csv_path = REPORT_DIR / 'validation_samples.csv'
    if csv_path.exists():
        df = pd.read_csv(csv_path)
        df = df.head(N_VALIDATION_SAMPLES).copy()
    else:
        train_zarrs = sorted((BASE_DIR / 'train').glob('*.zarr'))[:N_VALIDATION_SAMPLES]
        df = pd.DataFrame({'sample_id': [sample_id_from_path(p) for p in train_zarrs]})
    df['zarr_path'] = df['sample_id'].map(lambda s: BASE_DIR / 'train' / f'{s}.zarr')
    df['geff_path'] = df['sample_id'].map(lambda s: BASE_DIR / 'train' / f'{s}.geff')
    df['has_zarr'] = df['zarr_path'].map(lambda p: p.exists())
    df['has_geff'] = df['geff_path'].map(lambda p: p.exists())
    return df

validation_df = load_validation_samples()
display(validation_df[['sample_id', 'has_zarr', 'has_geff']])
assert validation_df['has_zarr'].all(), 'Some validation zarrs are missing'
assert validation_df['has_geff'].all(), 'Some validation geffs are missing'

## Local Scorer Functions

This is an approximate sparse-label scorer. It matches predicted and GT nodes per timepoint within `7 um`, then scores matched temporal edges. It is for comparing local variants, not a replacement for the official Kaggle metric.

In [ ]:
import zarr
from scipy.optimize import linear_sum_assignment

def read_geff(geff_path: Path):
    root = zarr.open(str(geff_path), mode='r')
    ids = np.asarray(root['nodes/ids'][:]).astype(int)
    nodes = pd.DataFrame({
        'node_id': ids,
        't': np.asarray(root['nodes/props/t/values'][:]).astype(int),
        'z': np.asarray(root['nodes/props/z/values'][:]).astype(float),
        'y': np.asarray(root['nodes/props/y/values'][:]).astype(float),
        'x': np.asarray(root['nodes/props/x/values'][:]).astype(float),
    })
    edges_arr = np.asarray(root['edges/ids'][:]).astype(int)
    edges = pd.DataFrame(edges_arr, columns=['source_id', 'target_id']) if edges_arr.size else pd.DataFrame(columns=['source_id', 'target_id'])
    est = np.nan
    meta_path = geff_path / 'zarr.json'
    if meta_path.exists():
        try:
            meta = json.loads(meta_path.read_text())
            attrs = meta.get('attributes', {})
            est = attrs.get('estimated_number_of_nodes', np.nan)
        except Exception:
            pass
    return nodes, edges, est

def physical_distance_matrix_um(a_zyx, b_zyx):
    a = np.asarray(a_zyx, dtype=float) * VOXEL_SCALE_UM
    b = np.asarray(b_zyx, dtype=float) * VOXEL_SCALE_UM
    diff = a[:, None, :] - b[None, :, :]
    return np.sqrt((diff * diff).sum(axis=2))

def match_pred_to_gt_by_time(pred_nodes, gt_nodes, max_distance_um=MATCH_MAX_DISTANCE_UM):
    rows = []
    pred_to_gt = {}
    gt_to_pred = {}
    for t in sorted(set(pred_nodes['t']).intersection(set(gt_nodes['t']))):
        p = pred_nodes[pred_nodes['t'] == t].reset_index(drop=True)
        g = gt_nodes[gt_nodes['t'] == t].reset_index(drop=True)
        if p.empty or g.empty:
            continue
        dist = physical_distance_matrix_um(p[['z','y','x']].to_numpy(), g[['z','y','x']].to_numpy())
        rr, cc = linear_sum_assignment(dist)
        for r, c in zip(rr, cc):
            d = float(dist[r, c])
            if d <= max_distance_um:
                pn = int(p.loc[r, 'node_id'])
                gn = int(g.loc[c, 'node_id'])
                rows.append({'t': int(t), 'pred_node_id': pn, 'gt_node_id': gn, 'distance_um': d})
                pred_to_gt[pn] = gn
                gt_to_pred[gn] = pn
    return pd.DataFrame(rows), pred_to_gt, gt_to_pred

def score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, estimated_number_of_nodes=np.nan):
    matches, pred_to_gt, gt_to_pred = match_pred_to_gt_by_time(pred_nodes, gt_nodes)
    gt_edge_set = set(map(tuple, gt_edges[['source_id','target_id']].astype(int).to_numpy())) if len(gt_edges) else set()
    edge_tp = 0
    matched_pred_edges = 0
    for row in pred_edges[['source_id','target_id']].astype(int).itertuples(index=False):
        s = pred_to_gt.get(int(row.source_id))
        t = pred_to_gt.get(int(row.target_id))
        if s is None or t is None:
            continue
        matched_pred_edges += 1
        if (s, t) in gt_edge_set:
            edge_tp += 1
    edge_fp_matched = max(0, matched_pred_edges - edge_tp)
    edge_fn = max(0, len(gt_edge_set) - edge_tp)
    edge_jaccard = edge_tp / max(1, edge_tp + edge_fp_matched + edge_fn)
    naive = edge_tp / max(1, edge_tp + max(0, len(pred_edges) - edge_tp) + edge_fn)
    if np.isfinite(estimated_number_of_nodes) and estimated_number_of_nodes > 0:
        over_factor = min(1.0, float(estimated_number_of_nodes) / max(1, len(pred_nodes)))
    else:
        over_factor = 1.0
    return {
        'pred_nodes': int(len(pred_nodes)),
        'gt_nodes_sparse': int(len(gt_nodes)),
        'matched_nodes': int(len(matches)),
        'node_recall_sparse_gt': len(matches) / max(1, len(gt_nodes)),
        'node_precision_sparse_gt': len(matches) / max(1, len(pred_nodes)),
        'pred_edges': int(len(pred_edges)),
        'gt_edges_sparse': int(len(gt_edge_set)),
        'matched_pred_edges': int(matched_pred_edges),
        'edge_tp': int(edge_tp),
        'edge_fp_matched': int(edge_fp_matched),
        'edge_fn': int(edge_fn),
        'edge_jaccard_matched': float(edge_jaccard),
        'edge_jaccard_naive': float(naive),
        'estimated_number_of_nodes': float(estimated_number_of_nodes) if np.isfinite(estimated_number_of_nodes) else np.nan,
        'node_overprediction_factor': float(over_factor),
        'edge_jaccard_adjusted_approx': float(edge_jaccard * over_factor),
        'match_distance_median_um': float(matches['distance_um'].median()) if len(matches) else np.nan,
        'match_distance_p95_um': float(matches['distance_um'].quantile(0.95)) if len(matches) else np.nan,
    }

def split_submission(submission_df, sample_id):
    sub = submission_df[submission_df['dataset'].astype(str) == sample_id].copy()
    nodes = sub[sub['row_type'] == 'node'][['node_id','t','z','y','x']].copy()
    edges = sub[sub['row_type'] == 'edge'][['source_id','target_id']].copy()
    return nodes, edges

print('scorer ready')

## Optional: Generate Baseline Local Submission

Set `RUN_BASELINE_PREDICTION=True` in the config cell to generate `reports/baseline_local_validation_submission.csv`.

In [ ]:
def build_submission_from_prediction_frames(prediction_frames):
    rows = []
    row_id = 0
    for sample_id, det, edges in prediction_frames:
        for r in det[['node_id','t','z','y','x']].itertuples(index=False):
            rows.append({
                'id': row_id, 'dataset': sample_id, 'row_type': 'node',
                'node_id': int(r.node_id), 't': int(r.t), 'z': int(round(r.z)), 'y': int(round(r.y)), 'x': int(round(r.x)),
                'source_id': -1, 'target_id': -1,
            })
            row_id += 1
        for r in edges[['source_id','target_id']].itertuples(index=False):
            rows.append({
                'id': row_id, 'dataset': sample_id, 'row_type': 'edge',
                'node_id': -1, 't': -1, 'z': -1, 'y': -1, 'x': -1,
                'source_id': int(r.source_id), 'target_id': int(r.target_id),
            })
            row_id += 1
    return pd.DataFrame(rows)

if RUN_BASELINE_PREDICTION:
    sys.path.insert(0, str(PROJECT_DIR))
    import biohub_baseline as bb
    bb.DETECTION_PARAMS.update({'percentile_high': 99.8, 'threshold_abs': 0.20, 'gaussian_sigma': 1.2, 'min_distance': 4})
    bb.LINKING_PARAMS.update({'max_distance_um': 9.0})
    preds = []
    for row in validation_df.itertuples(index=False):
        sample_id = row.sample_id
        img = bb.open_image_zarr(row.zarr_path, print_info=True)
        det = bb.detect_cells_for_sample(img, sample_id, bb.DETECTION_PARAMS, max_timepoints=BASELINE_MAX_TIMEPOINTS)
        edges = bb.link_detections(det, bb.LINKING_PARAMS)
        preds.append((sample_id, det, edges))
    baseline_sub = build_submission_from_prediction_frames(preds)
    out = METHOD_SUBMISSIONS['baseline']
    baseline_sub.to_csv(out, index=False)
    print('saved:', out, baseline_sub.shape)
else:
    print('RUN_BASELINE_PREDICTION=False; skipping baseline generation')

## Score Available Method CSVs

For DeepCenter, run the strong notebook on the same validation samples and save/copy its local output as `reports/deepcenter_local_validation_submission.csv`. Then rerun this scoring cell.

In [ ]:
def score_submission_file(method_name: str, submission_path: Path):
    if not submission_path.exists():
        print(f'{method_name}: missing {submission_path}')
        return pd.DataFrame()
    sub = pd.read_csv(submission_path)
    rows = []
    for row in validation_df.itertuples(index=False):
        sample_id = row.sample_id
        gt_nodes, gt_edges, est = read_geff(row.geff_path)
        pred_nodes, pred_edges = split_submission(sub, sample_id)
        if pred_nodes.empty:
            print(method_name, sample_id, 'has no predictions in CSV')
            continue
        score = score_one_sample(pred_nodes, pred_edges, gt_nodes, gt_edges, est)
        score.update({'method': method_name, 'sample_id': sample_id})
        rows.append(score)
    return pd.DataFrame(rows)

all_scores = []
for name, path in METHOD_SUBMISSIONS.items():
    df = score_submission_file(name, Path(path))
    if len(df):
        all_scores.append(df)

if all_scores:
    scores_df = pd.concat(all_scores, ignore_index=True)
    score_path = REPORT_DIR / 'method_compare_local_scores.csv'
    scores_df.to_csv(score_path, index=False)
    display(scores_df)
    metric_cols = ['edge_jaccard_adjusted_approx', 'edge_jaccard_matched', 'edge_jaccard_naive', 'node_recall_sparse_gt', 'pred_nodes', 'pred_edges', 'matched_nodes']
    summary = scores_df.groupby('method')[metric_cols].mean().sort_values('edge_jaccard_adjusted_approx', ascending=False)
    display(summary)
    print('saved:', score_path)
else:
    print('No method CSVs found yet. Generate baseline or copy DeepCenter local validation submission into reports/.')